# STEP 2: Feature Enrichment - Compute and Store Enhanced Features

**Purpose**: Compute all 37 enhanced features and store them in the database enrichments table

**What this notebook does**:
1. **Load Raw CVE Data** - Read cvss_vector, cwe, description from cves table
2. **Compute CVSS Features** - Parse CVSS vectors into 10 dimensional features
3. **Compute CWE Features** - Extract CWE intelligence (Top 25, categories, severity)
4. **Compute NLP Features** - Detect exploitation keywords in descriptions
5. **Compute Vendor Features** - Identify high-risk vendors
6. **Compute Interaction Features** - Create compound risk indicators
7. **Update Database** - Store all 37 computed features in enrichments table

**Input**: cves table (raw CVE data)
**Output**: enrichments table updated with 37 computed features

**Prerequisites**:
- STEP_1: Data ingestion completed
- STEP_2: External enrichments completed
- Database migration script executed (scripts/migrate_enrichments_schema.py)

---

## 1. Setup & Imports

In [12]:
import sys
from pathlib import Path
from datetime import datetime
import warnings

import pandas as pd
import numpy as np
from tqdm import tqdm

warnings.filterwarnings('ignore')

# Setup project paths
project_root = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(project_root))

# Import project modules
from src.core.cve_database import CVEDatabase
from src.features.enhanced_features import EnhancedFeatureExtractor
from config.settings import settings

print(f"[OK] Project root: {project_root}")
print(f"[OK] Database: {settings.get_database_path()}")
print(f"[OK] Imports successful")
print(f"\n[INFO] This notebook will compute 37 enhanced features and store them in the database")

[OK] Project root: /Users/vinayksharma/AirDnd/cti_recommender
[OK] Database: /Users/vinayksharma/AirDnd/cti_recommender/data/cve_database.db
[OK] Imports successful

[INFO] This notebook will compute 37 enhanced features and store them in the database


## 2. Database Connection & Validation

In [13]:
# Connect to database
db = CVEDatabase()

# Get statistics
stats = db.get_statistics()

print("="*70)
print("DATABASE STATUS")
print("="*70)
print(f"Total CVEs: {stats['total_cves']:,}")
print(f"\nEnrichment Coverage:")
print(f"  KEV entries: {stats.get('kev_count', 0):,}")
print(f"  EPSS scores: {stats.get('epss_count', 0):,}")
print(f"  Healthcare flags: {stats.get('healthcare_count', 0):,}")
print(f"  ATT&CK mappings: {stats.get('attack_count', 0):,}")
print(f"  CHPL entries: {stats.get('chpl_count', 0):,}")
print("="*70)

2026-03-28 20:51:20 - src.core.cve_database - INFO - Connected to database
2026-03-28 20:51:20 - src.core.cve_database - INFO - Database schema created/verified
DATABASE STATUS
Total CVEs: 226,320

Enrichment Coverage:
  KEV entries: 1,179
  EPSS scores: 0
  Healthcare flags: 2,009
  ATT&CK mappings: 0
  CHPL entries: 0


## 3. Load Raw CVE Data for Feature Computation

In [14]:
# Load CVEs with raw data needed for feature computation
query = """
SELECT 
    c.cve_id,
    c.cvss,
    c.cvss_vector,
    c.cwe,
    c.description,
    e.kev_flag,
    e.is_healthcare
FROM cves c
LEFT JOIN enrichments e ON c.cve_id = e.cve_id
WHERE c.cvss IS NOT NULL
ORDER BY c.cve_id
"""

print("[INFO] Loading CVE data for feature computation...")
df = pd.read_sql(query, db.conn)

print(f"\n[OK] Loaded {len(df):,} CVEs")
print(f"\nData availability:")
print(f"  CVEs with CVSS vector: {df['cvss_vector'].notna().sum():,} ({df['cvss_vector'].notna().mean()*100:.1f}%)")
print(f"  CVEs with CWE: {df['cwe'].notna().sum():,} ({df['cwe'].notna().mean()*100:.1f}%)")
print(f"  CVEs with description: {df['description'].notna().sum():,} ({df['description'].notna().mean()*100:.1f}%)")

# Display sample
print(f"\nSample data:")
df.head(3)

[INFO] Loading CVE data for feature computation...

[OK] Loaded 210,147 CVEs

Data availability:
  CVEs with CVSS vector: 210,147 (100.0%)
  CVEs with CWE: 208,046 (99.0%)
  CVEs with description: 210,147 (100.0%)

Sample data:


,cve_id,cvss,cvss_vector,cwe,description,kev_flag,is_healthcare
0,CVE-1999-0199,9.8,CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H,CWE-252,manual/search.texi in the GNU C Library (aka g...,0,0
1,CVE-2002-20001,7.5,CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:N/I:N/A:H,CWE-400,The Diffie-Hellman Key Agreement Protocol allo...,0,0
2,CVE-2002-20002,5.4,CVSS:3.1/AV:N/AC:H/PR:N/UI:N/S:C/C:L/I:L/A:N,CWE-338,The Net::EasyTCP package before 0.15 for Perl ...,0,0


## 4. Initialize Feature Extractor

In [15]:
# Initialize enhanced feature extractor
extractor = EnhancedFeatureExtractor()

print("[OK] EnhancedFeatureExtractor initialized")
print(f"\nFeature categories:")
print(f"  - CVSS Decomposition: 10 features")
print(f"  - CWE Intelligence: 8 features")
print(f"  - Description NLP: 10 features")
print(f"  - Vendor Features: 3 features")
print(f"  - Interaction Features: 6 features")
print(f"  Total: 37 computed features")

[OK] EnhancedFeatureExtractor initialized
  - CVSS decomposition: ready
  - CWE intelligence: ready
  - Description NLP: ready
  - Vendor intelligence: ready
  - Interaction features: ready
[OK] EnhancedFeatureExtractor initialized

Feature categories:
  - CVSS Decomposition: 10 features
  - CWE Intelligence: 8 features
  - Description NLP: 10 features
  - Vendor Features: 3 features
  - Interaction Features: 6 features
  Total: 37 computed features


## 5. Compute Enhanced Features

This step computes all 37 features for each CVE. This may take several minutes for large datasets.

In [16]:
print("="*70)
print("COMPUTING ENHANCED FEATURES")
print("="*70)
print(f"\n[INFO] Processing {len(df):,} CVEs...")
print(f"[INFO] This will compute 37 features per CVE")
print(f"[INFO] Estimated time: {len(df) / 1000:.1f} minutes (approximate)\n")

start_time = datetime.now()

# Apply enhanced feature extraction
enhanced_df = extractor.extract_all_features(df)

elapsed = (datetime.now() - start_time).total_seconds()

print(f"\n[OK] Feature computation completed in {elapsed:.1f} seconds ({elapsed/60:.1f} minutes)")
print(f"  Processing rate: {len(df)/elapsed:.0f} CVEs/second")
print(f"\n[INFO] Enhanced dataset shape: {enhanced_df.shape}")
print(f"  Total columns: {len(enhanced_df.columns)}")

# Show computed features sample
print(f"\nSample computed features:")
feature_cols = [c for c in enhanced_df.columns if c.startswith(('cvss_', 'cwe_', 'desc_', 'vendor_', 'ultimate', 'critical', 'network', 'auth', 'high_impact', 'healthcare_critical'))]
print(f"  Computed feature columns: {len(feature_cols)}")
enhanced_df[['cve_id'] + feature_cols[:5]].head(3)

COMPUTING ENHANCED FEATURES

[INFO] Processing 210,147 CVEs...
[INFO] This will compute 37 features per CVE
[INFO] Estimated time: 210.1 minutes (approximate)


ENHANCED FEATURE EXTRACTION
Extracting CVSS vector decomposition features...
  ✓ Added 8 CVSS decomposition features + 2 derived features
Extracting CWE intelligence features...
  ✓ Added 8 CWE intelligence features
Extracting description NLP features...
  ✓ Added 10 description NLP features
Extracting vendor intelligence features...
  ✓ Added 3 vendor intelligence features
Extracting enhanced interaction features...
  ✓ Added 6 interaction features

  Total new features added: 37

[OK] Feature computation completed in 5.1 seconds (0.1 minutes)
  Processing rate: 41478 CVEs/second

[INFO] Enhanced dataset shape: (210147, 44)
  Total columns: 44

Sample computed features:
  Computed feature columns: 38


,cve_id,cvss_vector,cvss_av,cvss_ac,cvss_pr,cvss_ui
0,CVE-1999-0199,CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H,4.0,2.0,3.0,2.0
1,CVE-2002-20001,CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:N/I:N/A:H,4.0,2.0,3.0,2.0
2,CVE-2002-20002,CVSS:3.1/AV:N/AC:H/PR:N/UI:N/S:C/C:L/I:L/A:N,4.0,1.0,3.0,2.0


## 6. Validate Computed Features

In [17]:
print("="*70)
print("FEATURE VALIDATION")
print("="*70)

# Check for computed features
computed_features = {
    'CVSS Decomposition': ['cvss_av', 'cvss_ac', 'cvss_pr', 'cvss_ui', 'cvss_s', 'cvss_c', 'cvss_i', 'cvss_a'],
    'CWE Intelligence': ['cwe_is_top25', 'cwe_is_injection', 'cwe_is_crypto'],
    'Description NLP': ['desc_has_rce', 'desc_has_sqli', 'desc_has_xss'],
    'Vendor Features': ['vendor_is_high_risk', 'vendor_is_healthcare'],
    'Interaction Features': ['ultimate_risk', 'critical_exploitable', 'network_accessible']
}

print(f"\nFeature Coverage:")
for category, features in computed_features.items():
    print(f"\n{category}:")
    for feat in features:
        if feat in enhanced_df.columns:
            non_null = enhanced_df[feat].notna().sum()
            pct = (non_null / len(enhanced_df)) * 100
            non_zero = (enhanced_df[feat] != 0).sum() if enhanced_df[feat].dtype in ['int64', 'float64'] else 0
            print(f"  {feat:30s}: {pct:5.1f}% coverage, {non_zero:6,} non-zero values")
        else:
            print(f"  {feat:30s}: [MISSING]")

print(f"\n" + "="*70)

FEATURE VALIDATION

Feature Coverage:

CVSS Decomposition:
  cvss_av                       : 100.0% coverage, 210,147 non-zero values
  cvss_ac                       : 100.0% coverage, 210,147 non-zero values
  cvss_pr                       : 100.0% coverage, 210,147 non-zero values
  cvss_ui                       : 100.0% coverage, 210,147 non-zero values
  cvss_s                        : 100.0% coverage, 210,147 non-zero values
  cvss_c                        : 100.0% coverage, 210,147 non-zero values
  cvss_i                        : 100.0% coverage, 210,147 non-zero values
  cvss_a                        : 100.0% coverage, 210,147 non-zero values

CWE Intelligence:
  cwe_is_top25                  : 100.0% coverage, 121,165 non-zero values
  cwe_is_injection              : 100.0% coverage, 51,695 non-zero values
  cwe_is_crypto                 : 100.0% coverage,  2,143 non-zero values

Description NLP:
  desc_has_rce                  : 100.0% coverage, 40,634 non-zero values
  desc_

## 7. Update Database with Computed Features

This step writes all computed features back to the enrichments table.

In [18]:
print("="*70)
print("UPDATING DATABASE")
print("="*70)
print(f"\n[INFO] Preparing to update enrichments table with computed features")
print(f"[INFO] CVEs to update: {len(enhanced_df):,}")

# Prepare feature columns for database update
feature_columns = [
    # CVSS Decomposition
    'cvss_av', 'cvss_ac', 'cvss_pr', 'cvss_ui', 'cvss_s', 
    'cvss_c', 'cvss_i', 'cvss_a', 'cvss_score_derived', 'cvss_severity_category',
    # CWE Intelligence
    'cwe_is_top25', 'cwe_is_injection', 'cwe_is_crypto', 'cwe_is_access_control',
    'cwe_is_input_validation', 'cwe_is_memory_corruption', 'cwe_category', 'cwe_severity_score',
    # Description NLP
    'desc_has_rce', 'desc_has_auth_bypass', 'desc_has_priv_esc', 'desc_has_sqli',
    'desc_has_xss', 'desc_has_dos', 'desc_has_buffer_overflow', 'desc_has_path_traversal',
    'desc_has_csrf', 'desc_has_xxe',
    # Vendor Features
    'vendor_is_high_risk', 'vendor_is_healthcare', 'vendor_risk_score',
    # Interaction Features
    'ultimate_risk', 'critical_exploitable', 'network_accessible', 
    'auth_not_required', 'high_impact_network', 'healthcare_critical'
]

# Filter to only include columns that exist in enhanced_df
available_features = [col for col in feature_columns if col in enhanced_df.columns]
print(f"[INFO] Available features to update: {len(available_features)} / {len(feature_columns)}")

if len(available_features) < len(feature_columns):
    missing = set(feature_columns) - set(available_features)
    print(f"[WARN] Missing features: {missing}")

UPDATING DATABASE

[INFO] Preparing to update enrichments table with computed features
[INFO] CVEs to update: 210,147
[INFO] Available features to update: 37 / 37


In [19]:
# Update database in batches
print(f"\n[INFO] Starting database update...")

batch_size = 1000
total_updated = 0
errors = 0

cursor = db.conn.cursor()

# Build UPDATE statement
set_clause = ", ".join([f"{col} = ?" for col in available_features])
update_sql = f"UPDATE enrichments SET {set_clause} WHERE cve_id = ?"

for i in tqdm(range(0, len(enhanced_df), batch_size), desc="Updating database"):
    batch = enhanced_df.iloc[i:i+batch_size]
    
    try:
        for _, row in batch.iterrows():
            values = [row.get(col) for col in available_features]
            values.append(row['cve_id'])  # Add cve_id for WHERE clause
            
            cursor.execute(update_sql, values)
            total_updated += 1
        
        # Commit batch
        db.conn.commit()
        
    except Exception as e:
        errors += 1
        print(f"\n[ERROR] Batch {i//batch_size + 1} failed: {e}")
        db.conn.rollback()

print(f"\n[OK] Database update completed")
print(f"  CVEs updated: {total_updated:,}")
print(f"  Errors: {errors}")
print(f"  Features per CVE: {len(available_features)}")


[INFO] Starting database update...


Updating database: 100%|██████████| 211/211 [00:06<00:00, 31.69it/s]


[OK] Database update completed
  CVEs updated: 210,147
  Errors: 0
  Features per CVE: 37


## 8. Verification - Query Updated Enrichments

In [20]:
print("="*70)
print("VERIFICATION")
print("="*70)

# Query enrichments table to verify updates
verify_query = """
SELECT 
    cve_id,
    kev_flag,
    cvss_av,
    cvss_s,
    cwe_is_top25,
    desc_has_rce,
    vendor_is_high_risk,
    ultimate_risk,
    network_accessible
FROM enrichments
WHERE cvss_av IS NOT NULL
LIMIT 10
"""

verify_df = pd.read_sql(verify_query, db.conn)

print(f"\n[INFO] Sample of updated enrichments:")
print(verify_df)

# Check coverage
coverage_query = """
SELECT 
    COUNT(*) as total,
    SUM(CASE WHEN cvss_av IS NOT NULL THEN 1 ELSE 0 END) as has_cvss_av,
    SUM(CASE WHEN cwe_is_top25 IS NOT NULL THEN 1 ELSE 0 END) as has_cwe_features,
    SUM(CASE WHEN desc_has_rce IS NOT NULL THEN 1 ELSE 0 END) as has_nlp_features,
    SUM(CASE WHEN ultimate_risk IS NOT NULL THEN 1 ELSE 0 END) as has_interaction_features
FROM enrichments
"""

coverage_df = pd.read_sql(coverage_query, db.conn)

print(f"\n[STATS] Enrichment Coverage:")
total = coverage_df['total'].iloc[0]
for col in coverage_df.columns[1:]:
    count = coverage_df[col].iloc[0]
    pct = (count / total) * 100 if total > 0 else 0
    print(f"  {col:30s}: {count:6,} / {total:,} ({pct:5.1f}%)")

print(f"\n" + "="*70)

VERIFICATION

[INFO] Sample of updated enrichments:
          cve_id  kev_flag  cvss_av  cvss_s  cwe_is_top25  desc_has_rce  \
0  CVE-2025-9994         0      4.0     1.0             1             0   
1  CVE-2025-9993         0      4.0     1.0             0             0   
2  CVE-2025-9992         0      4.0     2.0             1             0   
3  CVE-2025-9991         0      4.0     1.0             0             0   
4  CVE-2025-9990         0      4.0     1.0             0             0   
5  CVE-2025-9985         0      4.0     1.0             0             0   
6  CVE-2025-9984         0      4.0     1.0             1             0   
7  CVE-2025-9982         0      4.0     1.0             0             1   
8  CVE-2025-9981         0      4.0     2.0             1             0   
9  CVE-2025-9980         0      4.0     2.0             1             0   

   vendor_is_high_risk  ultimate_risk  network_accessible  
0                    0       3.000000                   1  
1 

## 9. Summary & Next Steps

In [21]:
print("="*70)
print("FEATURE ENRICHMENT SUMMARY")
print("="*70)
print(f"\n[OK] Feature enrichment completed successfully")
print(f"\n[STATS] Enrichment Results:")
print(f"  CVEs processed: {len(enhanced_df):,}")
print(f"  Features computed per CVE: {len(available_features)}")
print(f"  Database columns updated: {len(available_features)}")
print(f"  Processing time: {elapsed:.1f} seconds")

print(f"\n[ARCHITECTURE] Data Pipeline Status:")
print(f"  [OK] STEP_1: CVE Data Ingestion (raw NVD data)")
print(f"  [OK] STEP_2: External Enrichments (KEV, EPSS, Healthcare, ATT&CK, CHPL)")
print(f"  [OK] STEP_2: Feature Enrichment (37 computed features) <- CURRENT")
print(f"  [..] STEP_4: Feature Engineering + Labels")
print(f"  [..] STEP_5: Model Training + Scientific Protocol")
print(f"  [..] STEP_8: Advanced Models (artifact-aligned)")

print(f"\n[NEXT STEP] Run STEP_3_Feature_Engineering_Labels.ipynb")
print(f"\n[INFO] All 52 enrichment features are now available in the database:")
print(f"  - 15 external signals (KEV, EPSS, Healthcare, ATT&CK, CHPL, etc.)")
print(f"  - 37 computed features (CVSS, CWE, NLP, Vendor, Interactions)")
print(f"\n" + "="*70)

# Close database connection
db.conn.close()
print(f"\n[OK] Database connection closed")

FEATURE ENRICHMENT SUMMARY

[OK] Feature enrichment completed successfully

[STATS] Enrichment Results:
  CVEs processed: 210,147
  Features computed per CVE: 37
  Database columns updated: 37
  Processing time: 5.1 seconds

[ARCHITECTURE] Data Pipeline Status:
  [OK] STEP_1: CVE Data Ingestion (raw NVD data)
  [OK] STEP_2: External Enrichments (KEV, EPSS, Healthcare, ATT&CK, CHPL)
  [OK] STEP_2: Feature Enrichment (37 computed features) <- CURRENT
  [..] STEP_4: Feature Engineering + Labels
  [..] STEP_5: Model Training + Scientific Protocol
  [..] STEP_8: Advanced Models (artifact-aligned)

[NEXT STEP] Run STEP_3_Feature_Engineering_Labels.ipynb

[INFO] All 52 enrichment features are now available in the database:
  - 15 external signals (KEV, EPSS, Healthcare, ATT&CK, CHPL, etc.)
  - 37 computed features (CVSS, CWE, NLP, Vendor, Interactions)


[OK] Database connection closed


## 9. Academic Addendum: Feature QA and Distribution Diagnostics

This section adds formal feature quality checks (missingness, zero-rate, percentiles) and a compact distribution review artifact.

In [22]:
import numpy as np
import pandas as pd

print('\n' + '='*70)
print('FEATURE QA: COVERAGE, SPARSITY, AND DISTRIBUTION CHECKS')
print('='*70)

try:
    if 'enhanced_df' in globals() and isinstance(enhanced_df, pd.DataFrame) and not enhanced_df.empty:
        qa_df = enhanced_df.copy()
    elif 'df' in globals() and isinstance(df, pd.DataFrame) and not df.empty:
        qa_df = df.copy()
    else:
        raise ValueError('No in-memory feature dataframe found (`enhanced_df`/`df`).')

    non_feature_cols = {'cve_id', 'published', 'modified', 'cwe', 'cvss_vector', 'label', 'soft_label'}
    feature_cols_qa = [
        c for c in qa_df.columns
        if c not in non_feature_cols and pd.api.types.is_numeric_dtype(qa_df[c])
    ]
    if not feature_cols_qa:
        raise ValueError('No numeric feature columns found for QA.')

    rows = []
    for col in feature_cols_qa:
        s = pd.to_numeric(qa_df[col], errors='coerce')
        n = int(s.shape[0])
        missing_pct = float(s.isna().mean() * 100)
        zero_pct = float(((s.fillna(0) == 0).mean()) * 100)
        q05 = float(s.quantile(0.05)) if s.notna().any() else np.nan
        q50 = float(s.quantile(0.50)) if s.notna().any() else np.nan
        q95 = float(s.quantile(0.95)) if s.notna().any() else np.nan
        rows.append({
            'feature': col,
            'n_rows': n,
            'missing_pct': missing_pct,
            'zero_pct': zero_pct,
            'q05': q05,
            'q50': q50,
            'q95': q95,
        })

    feature_qa_df = pd.DataFrame(rows).sort_values(['missing_pct', 'zero_pct'], ascending=False)
    print(f"[INFO] Audited numeric features: {len(feature_qa_df)}")
    print('\nTop 15 by missingness/sparsity:')
    print(feature_qa_df.head(15).to_string(index=False))

    high_missing = feature_qa_df[feature_qa_df['missing_pct'] > 40]
    high_sparse = feature_qa_df[feature_qa_df['zero_pct'] > 95]
    print(f"\n[WARN] Features with >40% missing: {len(high_missing)}")
    print(f"[WARN] Features with >95% zeros: {len(high_sparse)}")

    out_path = project_root / 'outputs' / 'evaluation' / 'feature_qa_report.csv'
    out_path.parent.mkdir(parents=True, exist_ok=True)
    feature_qa_df.to_csv(out_path, index=False)
    print(f"[OK] Saved feature QA report -> {out_path}")

except Exception as e:
    print(f"[ERROR] Feature QA diagnostics failed: {e}")


FEATURE QA: COVERAGE, SPARSITY, AND DISTRIBUTION CHECKS
[INFO] Audited numeric features: 40

Top 15 by missingness/sparsity:
                 feature  n_rows  missing_pct  zero_pct  q05  q50  q95
     healthcare_critical  210147          0.0 99.635017  0.0  0.0  0.0
            desc_has_xxe  210147          0.0 99.544129  0.0  0.0  0.0
                kev_flag  210147          0.0 99.439916  0.0  0.0  0.0
           is_healthcare  210147          0.0 99.070175  0.0  0.0  0.0
           cwe_is_crypto  210147          0.0 98.980238  0.0  0.0  0.0
    vendor_is_healthcare  210147          0.0 98.867459  0.0  0.0  0.0
    desc_has_auth_bypass  210147          0.0 98.856515  0.0  0.0  0.0
 desc_has_path_traversal  210147          0.0 97.792973  0.0  0.0  0.0
       desc_has_priv_esc  210147          0.0 97.633085  0.0  0.0  0.0
           desc_has_csrf  210147          0.0 96.529096  0.0  0.0  0.0
 cwe_is_input_validation  210147          0.0 96.050860  0.0  0.0  0.0
desc_has_buffer_overfl

## 10. Feature Rationale (Academic Note)

Why these feature groups are used:
- CVSS features: severity and exploitability priors from standardized scoring.
- CWE structure features: weakness semantics and recurrence patterns.
- NLP text features: contextual signal from CVE descriptions.
- Vendor/product features: ecosystem concentration and historical exposure.
- Interaction features: non-linear combinations that capture risk compounding.

This rationale supports interpretability and motivates the feature set beyond pure trial-and-error.